In [1]:
import pandas as pd

# passengers table
passengers=pd.DataFrame({
    "Passenger_id":[1,2,3,4],
    "Name": ["Will", "Jane", "Alex", "Bill"],
    "Pclass":[3,1,2,1]
})

# a separate "extra info" table - notice PassengerId 5 doesn't exist in passengers,
# and PassengerId 4 doesn't exist here
extra_info=pd.DataFrame({
    "Passenger_id":[1,2,3,5],
    "Cabin":["C85", "C123", None, "B42"]
})

print(passengers)
print(extra_info)

   Passenger_id  Name  Pclass
0             1  Will       3
1             2  Jane       1
2             3  Alex       2
3             4  Bill       1
   Passenger_id Cabin
0             1   C85
1             2  C123
2             3   NaN
3             5   B42


In [2]:
#inner join - only keeps rows where Passengerid exists in both tables
inner=passengers.merge(extra_info,on="Passenger_id",how="inner")
print("Inner: \n",inner)

#left join- keeps all rows from passengers table ,fills missing info with NaN
left=passengers.merge(extra_info,on="Passenger_id",how="left")
print("Left: \n",left)

#right join - keep all rows from extra_info ,fill missing info with NaN
right=passengers.merge(extra_info,on="Passenger_id",how="right")
print("Right: \n",right)

#outer join - keeps everything from both tables
outer=passengers.merge(extra_info,on="Passenger_id",how="outer")
print("Outer: \n",outer)



Inner: 
    Passenger_id  Name  Pclass Cabin
0             1  Will       3   C85
1             2  Jane       1  C123
2             3  Alex       2   NaN
Left: 
    Passenger_id  Name  Pclass Cabin
0             1  Will       3   C85
1             2  Jane       1  C123
2             3  Alex       2   NaN
3             4  Bill       1   NaN
Right: 
    Passenger_id  Name  Pclass Cabin
0             1  Will     3.0   C85
1             2  Jane     1.0  C123
2             3  Alex     2.0   NaN
3             5   NaN     NaN   B42
Outer: 
    Passenger_id  Name  Pclass Cabin
0             1  Will     3.0   C85
1             2  Jane     1.0  C123
2             3  Alex     2.0   NaN
3             4  Bill     1.0   NaN
4             5   NaN     NaN   B42


In [8]:
#join-works only when our key column is index ,otherwise dosent work
passengers_idx=passengers.set_index("Passenger_id")
extra_info_idx=extra_info.set_index("Passenger_id")
join=passengers_idx.join(extra_info_idx)
print(join)

              Name  Pclass Cabin
Passenger_id                    
1             Will       3   C85
2             Jane       1  C123
3             Alex       2   NaN
4             Bill       1   NaN


In [3]:
record=pd.read_csv("Titanic-Dataset.csv")
print(record.columns)

# split into two fake "separate tables" just for practice
basic_info=record[["PassengerId","Name","Sex","Age"]]
survival_info=record[["PassengerId","Survived","Pclass"]]

# merge them back together
merged=basic_info.merge(survival_info,on="PassengerId",how="inner")
print(merged.head())
print(merged.shape)
print(record.shape)


Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')
   PassengerId                                               Name     Sex  \
0            1                            Braund, Mr. Owen Harris    male   
1            2  Cumings, Mrs. John Bradley (Florence Briggs Th...  female   
2            3                             Heikkinen, Miss. Laina  female   
3            4       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female   
4            5                           Allen, Mr. William Henry    male   

    Age  Survived  Pclass  
0  22.0         0       3  
1  38.0         1       1  
2  26.0         1       3  
3  35.0         1       1  
4  35.0         0       3  
(891, 6)
(891, 12)


In [9]:
#concat stacks dataframe-useful when we have same columns ,different rows
batch1=record.iloc[0:400]
batch2=record.iloc[400:891]
combined=pd.concat([batch1,batch2])
combined_inner=pd.concat([batch1,batch2],join="inner")
print(combined.head())
print(combined_inner.head())
print(combined.shape)

# reset index since it'll have gaps/duplicates after concat
combined=combined.reset_index(drop=True)


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked     Sex_Pclass  
0      0         A/5 21171   7.2500   NaN        S    male_class3  
1      0          PC 17599  71.2833   C85        C  female_class1  
2      0  STON/O2. 3101282   7.9250   NaN        S  female_class3  
3      0            113803  53.1000  C123       

In [6]:
# rename columns to be cleaner/more consistent
record_renamed=record.rename(columns={"Pclass":"PassengerClass",
                                      "SibSp":"Siblings"})
print(record_renamed.columns)

# rename the index itself (rare, but useful to know)
record_renamed=record_renamed.rename_axis("Passenger_index")
print(record_renamed.head())

# combine two columns into one new descriptive column
record["Sex_Pclass"]=record["Sex"]+ "_class"+ record["Pclass"].astype(str)
print(record[["Sex","Pclass","Sex_Pclass"]].head())

Index(['PassengerId', 'Survived', 'PassengerClass', 'Name', 'Sex', 'Age',
       'Siblings', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')
                 PassengerId  Survived  PassengerClass  \
Passenger_index                                          
0                          1         0               3   
1                          2         1               1   
2                          3         1               3   
3                          4         1               1   
4                          5         0               3   

                                                              Name     Sex  \
Passenger_index                                                              
0                                          Braund, Mr. Owen Harris    male   
1                Cumings, Mrs. John Bradley (Florence Briggs Th...  female   
2                                           Heikkinen, Miss. Laina  female   
3                     Futrelle, Mrs. Jacqu